# NuCLS · Entregable 2 · Modelos adicionales

**Nuevo bloque exploratorio. No modifica el Entregable 1 ni evalúa la reserva.**

Diez candidatos de cuatro familias. Primero se exportan las evidencias base y se mide el costo de GPU; luego se pueden ejecutar búsqueda (40 corridas) y piloto (50 corridas). Cinco semillas son el piloto para calcular potencia, no el benchmark final. La selección bibliográfica definitiva, replicación y análisis de robustez/interpretabilidad siguen pendientes.

Abre en Safari, guarda una copia en Drive y elige GPU T4 (o registra otra GPU para todo este bloque). Ejecuta por pasos. **No pulses Ejecutar todo inicialmente**: primero revisa el tiempo estimado del perfil. No ejecutes dos copias simultáneas. En una sesión nueva, ejecuta otra vez preparación; el entrenamiento omitirá las corridas completas y retomará la última época guardada.


## 1 · Conectar Drive y preparar el código incluido
No necesitas subir un ZIP de código. Usa la cuenta donde quedó el cierre de 22 semillas.

In [ ]:
import os, sys, json, base64, hashlib, zipfile, io, subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
BASES = Path('/content/drive/MyDrive/NuCLS_E2_CONTINUACION_BASES_22')
INDICE = BASES / 'indice_validacion.csv'
if not INDICE.is_file():
    INDICE = BASES / 'final/indice_validacion.csv'
assert INDICE.is_file(), 'No aparece el cierre de 22 semillas: conecta la misma cuenta de Drive.'
CODIGO = Path('/content/nucls_e2_adicionales_codigo')
DATOS = Path('/content/nucls_e2_datos')
SALIDA = Path('/content/drive/MyDrive/NuCLS_E2_ADICIONALES_EXPLORATORIO')
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '1e26743076b4f2406fde42ffe12c0314e7c699e7737da966d52fb6e5ae23d27e'
CODIGO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODIGO / name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
print('Código verificado. Resultados nuevos:', SALIDA)


## 2 · Bibliotecas y datos
Se conserva PyTorch instalado. La comparación exige el entorno del bloque base; si cambió, detente y comparte la salida. Solo se extraen train y validación.

In [ ]:
import csv, platform
import torch, torchvision, numpy as np
assert torch.cuda.is_available(), 'Selecciona GPU en Entorno de ejecución → Cambiar tipo.'
filas = list(csv.DictReader(INDICE.open()))
original = next(Path(r['ruta']).parents[1] for r in filas if 'NuCLS_V2_RESULTADOS_pool_corregido' in r['ruta'])
cfg = json.loads((original / 'configuracion.json').read_text())
actual = {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__}
print('Entorno actual:', actual)
for k, v in actual.items():
    assert v == cfg[k], f'Cambió {k}: actual {v}; anterior {cfg[k]}. No mezclar sin documentar.'
assert platform.python_version_tuple()[:2] == tuple(cfg['python'].split()[0].split('.')[:2]), 'Cambió la versión mayor/menor de Python.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'timm==1.0.22'], check=True)
import timm
assert timm.__version__ == '1.0.22', 'Reinicia la sesión si habías importado otra versión de timm.'
print('GPU:', torch.cuda.get_device_name(0), '| timm:', timm.__version__)
if not (DATOS / 'manifiesto.csv').is_file():
    fuente = Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
    assert fuente.is_file(), 'No aparece NuCLS_V2_DATOS.zip en este Drive.'
    subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/preparar_datos.py'), '--zip', str(fuente), '--salida', str(DATOS)], check=True)
print('Preparación lista. El perfil verificará el manifiesto y las imágenes.')


## 3 · Reunir las evidencias de las 66 corridas
Lee las carpetas originales sin modificarlas. Genera un ZIP nuevo sin pesos para completar estadística por paciente y curvas.

In [ ]:
subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/exportar_bases.py'), '--indice', str(INDICE), '--salida', '/content/drive/MyDrive/NuCLS_E2_BASES_66_EVIDENCIAS.zip'], check=True)


## 4 · Medir compatibilidad, memoria y velocidad
12 pasos por modelo con un lote real. No es el entrenamiento completo. Al finalizar se imprime una estimación para 40 ajustes + 50 corridas piloto; falta añadir validación y escritura en Drive.

In [ ]:
import datetime, signal
COMANDO = [sys.executable, '-u', str(CODIGO/'scripts/adicionales_e2.py'), '--datos', str(DATOS), '--salida', str(SALIDA)]
def ejecutar(fase):
    SALIDA.mkdir(parents=True, exist_ok=True)
    log = SALIDA / ('log_' + fase + '_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.txt')
    with log.open('w') as archivo:
        proceso = subprocess.Popen(COMANDO + ['--fase', fase], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for linea in proceso.stdout:
                print(linea, end='', flush=True); archivo.write(linea); archivo.flush()
            retorno = proceso.wait()
        except KeyboardInterrupt:
            proceso.send_signal(signal.SIGINT)
            try: proceso.wait(timeout=60)
            except subprocess.TimeoutExpired:
                proceso.terminate()
                proceso.wait(timeout=30)
            raise
    print('Log:', log)
    assert retorno == 0, 'La fase no terminó: comparte las últimas líneas. No borres resultados.'
ejecutar('perfil')
print((SALIDA/'presupuesto.json').read_text())


## 5 · Búsqueda y piloto (ejecución larga)
**Ejecuta después de revisar el perfil.** Cuatro configuraciones iguales por modelo, luego cinco semillas por modelo. Guardado por época y reanudación. Si reiniciaste la sesión, ejecuta primero pasos 1, 2 y 4 para recuperar las variables. No abre la reserva. El piloto no completa por sí solo todos los requisitos del Entregable 2.

In [ ]:
ejecutar('busqueda')
ejecutar('piloto')


## 6 · Referencias simples
Clase mayoritaria y regresión logística con cuatro valores de regularización, usando entrenamiento/validación. Se ejecutan después de la celda larga; no en otra sesión simultánea.

In [ ]:
subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/baselines_simples.py'), '--datos', str(DATOS), '--salida', '/content/drive/MyDrive/NuCLS_E2_REFERENCIAS_SIMPLES'], check=True)


## 7 · Consultar estado y descargar evidencias
Esta celda es independiente: funciona aunque se hayan perdido las variables de la sesión. Guarda también Archivo → Descargar → .ipynb con las salidas. Los pesos permanecen en Drive; no se incluyen en ZIP.

In [ ]:
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
carpeta = Path('/content/drive/MyDrive/NuCLS_E2_ADICIONALES_EXPLORATORIO')
estado = carpeta/'estado.json'
print(estado.read_text() if estado.exists() else 'Piloto aún no cerrado. Consulta el último log; no significa pérdida de resultados.')
for nombre in ['NuCLS_E2_BASES_66_EVIDENCIAS.zip', 'NuCLS_E2_ADICIONALES_EXPLORATORIO_RESULTADOS.zip', 'NuCLS_E2_REFERENCIAS_SIMPLES_RESULTADOS.zip']:
    ruta = carpeta.parent/nombre
    print(nombre, 'DISPONIBLE' if ruta.is_file() else 'AÚN NO GENERADO')
# Descarga primero el respaldo de las bases; los demás pueden bajarse desde Drive.
respaldo = carpeta.parent/'NuCLS_E2_BASES_66_EVIDENCIAS.zip'
if respaldo.is_file(): files.download(str(respaldo))
